# Taller de Big Data: Pipeline ETL Distribuido con PySpark y MySQL

**Módulo:** Big Data y Procesamiento Distribuido  

---

## Objetivo
Construir un **pipeline de datos** de extremo a extremo que:

1. **Extrae** un dataset masivo (≥ 50,000 transacciones e-commerce en CSV).
2. **Transforma** los datos de forma distribuida con **PySpark** (limpieza, columnas calculadas, filtros y agregaciones).
3. **Carga** las métricas analíticas en **MySQL** mediante sentencias preparadas y valida la persistencia con un `SELECT` final.

```
 CSV (60K filas)  SparkSession  Limpieza  Columnas calculadas  groupBy  MySQL (InnoDB)
     Extract                     Transform (distribuido)                               Load + Validación
```

**Entregable:**
| Archivo | Descripción |
|---|---|
| `Minero_Guillermo_Taller_BigData.ipynb` | Notebook autosuficiente (Genera dataset, procesa en PySpark y persiste en MySQL) |
| `script_mysql_bigdata.sql` | DDL, usuario, tablas y consultas de validación para MySQL Workbench |

---

## Fase 1: Configuración del Entorno y Extracción

### Paso 1.1: Instalación de dependencias
- `pyspark` → motor de procesamiento distribuido (requiere **Java 17+** en el `PATH`).
- `mysql-connector-python` → driver oficial de Oracle para conectar Python con MySQL.

In [ ]:
%pip install -q pyspark mysql-connector-python pandas matplotlib

### Paso 1.2: Importación de librerías y variables de entorno
En Windows se fija `PYSPARK_PYTHON` para que los *workers* de Spark usen el mismo intérprete que el kernel de Jupyter.

In [ ]:
import os
import sys
import subprocess
from decimal import Decimal
from getpass import getpass

import pandas as pd
import matplotlib.pyplot as plt

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType

import mysql.connector
from mysql.connector import Error as MySQLError

# Los workers de Spark usan el mismo Python que el kernel (evita errores en Windows)
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

print(f"Python      : {sys.version.split()[0]}")
print(f"mysql-conn  : {mysql.connector.__version__}")

### Paso 1.3: Inicialización de una SparkSession optimizada
| Configuración | Propósito |
|---|---|
| `appName` | Nombre descriptivo visible en la Spark UI (`http://localhost:4040`) |
| `master("local[*]")` | Usa todos los núcleos de la máquina como executors paralelos |
| `spark.sql.shuffle.partitions = 8` | Reduce las 200 particiones por defecto (overhead innecesario para ~60K filas) |
| `spark.sql.adaptive.enabled` | **AQE**: re-optimiza el plan en tiempo de ejecución (coalesce de particiones, skew joins) |
| `spark.serializer = Kryo` | Serialización más rápida y compacta que la de Java |
| `spark.driver.memory = 2g` | Memoria suficiente para `collect()` / `toPandas()` de resultados agregados |

In [ ]:
spark = (
    SparkSession.builder
    .appName("Pipeline_ETL_Transacciones_Ecommerce_MySQL")
    .master("local[*]")
    .config("spark.driver.memory", "2g")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.sql.adaptive.enabled", "true")
    .config("spark.sql.adaptive.coalescePartitions.enabled", "true")
    .config("spark.serializer", "org.apache.spark.serializer.KryoSerializer")
    .config("spark.sql.session.timeZone", "America/El_Salvador")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")

print(f"Spark versión     : {spark.version}")
print(f"Aplicación        : {spark.sparkContext.appName}")
print(f"Paralelismo (cores): {spark.sparkContext.defaultParallelism}")

### Paso 1.4: Generación y Creación del Dataset Masivo (CSV)
Dataset sintético de **transacciones e-commerce** (El Salvador, año 2025) con **60,600 filas** (supera los 50,000 solicitados).  
Contiene intencionalmente **nulos, duplicados, cantidades negativas, precios en cero y categorías con formato inconsistente** para demostrar la fase de limpieza distribuida.

Se genera directamente desde el código en el notebook con una semilla determinista (`seed=42`).

In [ ]:
import csv
import random
from datetime import datetime, timedelta

RUTA_CSV = "transacciones_ecommerce.csv"

def generar_dataset_csv(ruta_archivo=RUTA_CSV, n_registros=60000):
    random.seed(42)
    catalogo = {
        "Electrónica": [("Laptop", 850.0), ("Smartphone", 520.0), ("Audífonos", 75.0), ("Monitor", 210.0), ("Tablet", 330.0)],
        "Hogar":       [("Licuadora", 65.0), ("Cafetera", 90.0), ("Aspiradora", 180.0), ("Lámpara", 35.0)],
        "Ropa":        [("Camisa", 28.0), ("Pantalón", 40.0), ("Zapatos", 70.0), ("Chaqueta", 95.0)],
        "Deportes":    [("Bicicleta", 420.0), ("Balón", 25.0), ("Pesas", 60.0), ("Tenis Running", 110.0)],
        "Libros":      [("Novela", 18.0), ("Libro Técnico", 55.0), ("Cómic", 12.0)],
    }
    metodos_pago = ["Tarjeta Crédito", "Tarjeta Débito", "PayPal", "Transferencia", "Efectivo"]
    ciudades = ["San Salvador", "Santa Ana", "San Miguel", "La Libertad", "Sonsonate", "Usulután"]
    estados = ["Completada"] * 8 + ["Cancelada", "Reembolsada"]
    pesos_hora = [1, 1, 1, 1, 1, 2, 3, 5, 7, 8, 9, 11, 13, 11, 9, 8, 8, 9, 11, 13, 14, 12, 7, 3]
    inicio = datetime(2025, 1, 1)

    def ensuciar_categoria(cat: str) -> str:
        r = random.random()
        if r < 0.04: return cat.upper()
        if r < 0.08: return f"  {cat.lower()} "
        return cat

    filas = []
    for i in range(1, n_registros + 1):
        cat = random.choice(list(catalogo))
        producto, precio_base = random.choice(catalogo[cat])
        hora = random.choices(range(24), weights=pesos_hora)[0]
        fecha = inicio + timedelta(days=random.randint(0, 364), hours=hora,
                                   minutes=random.randint(0, 59), seconds=random.randint(0, 59))
        precio = round(precio_base * random.uniform(0.85, 1.15), 2)
        cantidad = random.choices([1, 2, 3, 4, 5], weights=[50, 25, 12, 8, 5])[0]

        fila = {
            "transaccion_id": f"TX{i:06d}",
            "fecha_hora": fecha.strftime("%Y-%m-%d %H:%M:%S"),
            "cliente_id": f"C{random.randint(1, 8000):05d}",
            "producto": producto,
            "categoria": ensuciar_categoria(cat),
            "cantidad": cantidad,
            "precio_unitario": precio,
            "metodo_pago": random.choice(metodos_pago),
            "ciudad": random.choice(ciudades),
            "estado": random.choice(estados),
        }

        # Inyección controlada de ruido
        r = random.random()
        if r < 0.015: fila["precio_unitario"] = ""
        elif r < 0.025: fila["cantidad"] = ""
        elif r < 0.035: fila["cantidad"] = -cantidad
        elif r < 0.040: fila["precio_unitario"] = 0
        elif r < 0.050: fila["metodo_pago"] = ""
        elif r < 0.055: fila["fecha_hora"] = ""
        filas.append(fila)

    # Duplicados intencionales (~1%)
    filas.extend(random.sample(filas, 600))
    random.shuffle(filas)

    with open(ruta_archivo, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=list(filas[0].keys()))
        writer.writeheader()
        writer.writerows(filas)
    print(f"Dataset generado con éxito: '{ruta_archivo}' -> {len(filas):,} registros.")

# Generar archivo si no existe
if not os.path.exists(RUTA_CSV):
    generar_dataset_csv(RUTA_CSV)
else:
    print(f"Archivo '{RUTA_CSV}' encontrado.")

### Paso 1.5: Ingesta y Extracción Masiva con PySpark
Se define un esquema formal explícito (`StructType`) para optimizar el tiempo de lectura y evitar escaneos dobles innecesarios con `inferSchema`.

In [ ]:
esquema = StructType([
    StructField("transaccion_id",  StringType(),  nullable=False),
    StructField("fecha_hora",      StringType(),  nullable=True),
    StructField("cliente_id",      StringType(),  nullable=True),
    StructField("producto",        StringType(),  nullable=True),
    StructField("categoria",       StringType(),  nullable=True),
    StructField("cantidad",        IntegerType(), nullable=True),
    StructField("precio_unitario", DoubleType(),  nullable=True),
    StructField("metodo_pago",     StringType(),  nullable=True),
    StructField("ciudad",          StringType(),  nullable=True),
    StructField("estado",          StringType(),  nullable=True),
])

df_raw = (
    spark.read
    .option("header", "true")
    .option("encoding", "UTF-8")
    .option("mode", "PERMISSIVE")
    .schema(esquema)
    .csv(RUTA_CSV)
)

total_raw = df_raw.count()
print(f"Registros cargados en PySpark : {total_raw:,}")
print(f"Número de particiones RDD     : {df_raw.rdd.getNumPartitions()}")
df_raw.printSchema()
df_raw.show(5, truncate=False)

---

## Fase 2: Procesamiento Distribuido con PySpark

### Paso 2.1: Perfilado de calidad de datos
Antes de limpiar, se cuantifican los problemas: nulos por columna, duplicados y valores fuera de rango.

In [ ]:
# Nulos por columna (una sola pasada distribuida sobre el dataset)
print("=== Valores nulos por columna ===")
df_raw.select([F.sum(F.col(c).isNull().cast("int")).alias(c) for c in df_raw.columns]).show(truncate=False)

duplicados = total_raw - df_raw.dropDuplicates().count()
print(f"Filas duplicadas exactas : {duplicados:,}")
print(f"Cantidades <= 0          : {df_raw.filter(F.col('cantidad') <= 0).count():,}")
print(f"Precios <= 0             : {df_raw.filter(F.col('precio_unitario') <= 0).count():,}")

print("\n=== Inconsistencias de formato en 'categoria' ===")
df_raw.groupBy("categoria").count().orderBy("categoria").show(truncate=False)

### Paso 2.2: Limpieza de nulos e inconsistencias
| Regla | Tratamiento |
|---|---|
| Duplicados exactos / por `transaccion_id` | `dropDuplicates()` |
| `categoria` con espacios y mayúsculas mixtas | `trim()` + `initcap()` |
| `fecha_hora` texto | `to_timestamp()` (formato inválido → null) |
| `metodo_pago` nulo (no crítico) | Imputación → `"Desconocido"` |
| `fecha_hora`, `cantidad`, `precio_unitario` nulos (críticos) | Eliminación de la fila |
| `cantidad <= 0` o `precio_unitario <= 0` | Filtro de valores inconsistentes |

In [ ]:
df_clean = (
    df_raw
    .dropDuplicates()                                    # duplicados exactos
    .dropDuplicates(["transaccion_id"])                  # garantiza unicidad de la clave de negocio
    .withColumn("categoria", F.initcap(F.trim(F.col("categoria"))))
    .withColumn("fecha_hora", F.to_timestamp("fecha_hora", "yyyy-MM-dd HH:mm:ss"))
    .fillna({"metodo_pago": "Desconocido"})
    .dropna(subset=["fecha_hora", "cantidad", "precio_unitario", "categoria"])
    .filter((F.col("cantidad") > 0) & (F.col("precio_unitario") > 0))
)

total_clean = df_clean.count()
print(f"Registros originales : {total_raw:,}")
print(f"Registros limpios    : {total_clean:,}")
print(f"Registros descartados: {total_raw - total_clean:,} ({(total_raw - total_clean) / total_raw:.2%})")

print("\n=== Categorías normalizadas ===")
df_clean.groupBy("categoria").count().orderBy("categoria").show()

### Paso 2.3: Columnas calculadas
Se aplica el **IVA de El Salvador (13 %)** y se derivan atributos temporales y de segmentación:

- `subtotal` = `cantidad × precio_unitario`
- `impuesto` = `subtotal × 0.13`
- **`total_con_impuestos`** = `subtotal + impuesto`
- `fecha`, `hora`, `dia_semana` → extraídos del timestamp
- `segmento_ticket` → **Alto** (≥ $500), **Medio** (≥ $100), **Bajo**

El DataFrame resultante se guarda en caché (`cache()`) porque se reutiliza en varias agregaciones.

In [ ]:
TASA_IVA = 0.13

dias_es = F.create_map(*[F.lit(x) for x in (
    1, "Domingo", 2, "Lunes", 3, "Martes", 4, "Miércoles", 5, "Jueves", 6, "Viernes", 7, "Sábado")])

df_enriched = (
    df_clean
    .withColumn("subtotal",            F.round(F.col("cantidad") * F.col("precio_unitario"), 2))
    .withColumn("impuesto",            F.round(F.col("subtotal") * F.lit(TASA_IVA), 2))
    .withColumn("total_con_impuestos", F.round(F.col("subtotal") + F.col("impuesto"), 2))
    .withColumn("fecha",               F.to_date("fecha_hora"))
    .withColumn("hora",                F.hour("fecha_hora"))
    .withColumn("dia_semana",          dias_es[F.dayofweek("fecha_hora")])
    .withColumn("segmento_ticket",
                F.when(F.col("total_con_impuestos") >= 500, "Alto")
                 .when(F.col("total_con_impuestos") >= 100, "Medio")
                 .otherwise("Bajo"))
    .cache()
)

df_enriched.select("transaccion_id", "fecha_hora", "hora", "dia_semana", "categoria", "cantidad",
                   "precio_unitario", "subtotal", "impuesto", "total_con_impuestos", "segmento_ticket") \
           .show(8, truncate=False)

### Paso 2.4: Filtros avanzados
1. **Ventas efectivas:** solo transacciones `Completada` (se excluyen canceladas y reembolsadas) → base para las métricas.
2. **Filtro compuesto de negocio:** compras de **alto valor** (`total ≥ $500`) en categorías **Electrónica / Deportes**, durante el **pico nocturno (18:00–22:59)**, con método de pago identificado.

In [ ]:
# 1) Base analítica: solo ventas completadas
df_ventas = df_enriched.filter(F.col("estado") == "Completada")
print(f"Ventas completadas : {df_ventas.count():,} de {total_clean:,}")

# 2) Filtro avanzado multicondición
df_premium_nocturno = df_ventas.filter(
    (F.col("total_con_impuestos") >= 500)
    & (F.col("categoria").isin("Electrónica", "Deportes"))
    & (F.col("hora").between(18, 22))
    & (F.col("metodo_pago") != "Desconocido")
)
print(f"Ventas premium en pico nocturno: {df_premium_nocturno.count():,}")
df_premium_nocturno.select("transaccion_id", "fecha_hora", "categoria", "producto",
                           "metodo_pago", "total_con_impuestos") \
                   .orderBy(F.desc("total_con_impuestos")).show(10, truncate=False)

print("=== Distribución por estado y segmento ===")
df_enriched.groupBy("estado").pivot("segmento_ticket", ["Bajo", "Medio", "Alto"]).count().show()

### Paso 2.5: Agregaciones con `groupBy` (métricas clave)
**A) Métricas por hora del día** → conteo de transacciones, clientes únicos, unidades, ticket promedio, ingresos e impuestos.  
**B) Métricas por categoría** → incluye ticket máximo y **participación % en ingresos** (calculada con una ventana global).

Estas dos tablas agregadas son las que se persistirán en MySQL.

In [ ]:
from pyspark.sql.window import Window

# A) Métricas por hora
metricas_hora = (
    df_ventas.groupBy("hora")
    .agg(
        F.count("*").alias("total_transacciones"),
        F.countDistinct("cliente_id").alias("clientes_unicos"),
        F.sum("cantidad").alias("unidades_vendidas"),
        F.round(F.avg("total_con_impuestos"), 2).alias("ticket_promedio"),
        F.round(F.sum("total_con_impuestos"), 2).alias("ingresos_totales"),
        F.round(F.sum("impuesto"), 2).alias("impuestos_recaudados"),
    )
    .orderBy("hora")
)

# B) Métricas por categoría (+ participación porcentual)
ventana_global = Window.partitionBy()
metricas_categoria = (
    df_ventas.groupBy("categoria")
    .agg(
        F.count("*").alias("total_transacciones"),
        F.sum("cantidad").alias("unidades_vendidas"),
        F.round(F.avg("total_con_impuestos"), 2).alias("ticket_promedio"),
        F.round(F.max("total_con_impuestos"), 2).alias("ticket_maximo"),
        F.round(F.sum("total_con_impuestos"), 2).alias("ingresos_totales"),
    )
    .withColumn("participacion_pct",
                F.round(F.col("ingresos_totales") / F.sum("ingresos_totales").over(ventana_global) * 100, 2))
    .orderBy(F.desc("ingresos_totales"))
)

print("=== A) Métricas por hora ===")
metricas_hora.show(24)
print("=== B) Métricas por categoría ===")
metricas_categoria.show()

### Paso 2.6: Visualización de los resultados agregados
Los resultados agregados son pequeños (24 y 5 filas), por lo que es seguro traerlos al driver con `toPandas()`.

In [ ]:
pdf_hora = metricas_hora.toPandas()
pdf_cat = metricas_categoria.toPandas()

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

axes[0].bar(pdf_hora["hora"], pdf_hora["total_transacciones"], color="#3498db", edgecolor="black")
ax2 = axes[0].twinx()
ax2.plot(pdf_hora["hora"], pdf_hora["ticket_promedio"], color="#e74c3c", marker="o", linewidth=2)
axes[0].set_title("Transacciones y Ticket Promedio por Hora", fontweight="bold")
axes[0].set_xlabel("Hora del día"); axes[0].set_ylabel("N.º transacciones"); ax2.set_ylabel("Ticket promedio ($)")
axes[0].set_xticks(range(24))

axes[1].barh(pdf_cat["categoria"], pdf_cat["ingresos_totales"], color="#2ecc71", edgecolor="black")
for i, (v, p) in enumerate(zip(pdf_cat["ingresos_totales"], pdf_cat["participacion_pct"])):
    axes[1].text(v, i, f"  ${v:,.0f} ({p}%)", va="center", fontsize=9)
axes[1].set_title("Ingresos Totales por Categoría (con IVA)", fontweight="bold")
axes[1].set_xlabel("Ingresos ($)"); axes[1].invert_yaxis()
axes[1].set_xlim(0, pdf_cat["ingresos_totales"].max() * 1.3)

plt.tight_layout()
plt.show()

---

## Fase 3: Persistencia en MySQL

### Paso 3.1: Conexión segura Python → MySQL
Buenas prácticas aplicadas:
- **Sin credenciales en el código**: se leen de variables de entorno (`MYSQL_USER`, `MYSQL_PASSWORD`, …) o se solicitan con `getpass` (no se muestran en pantalla ni quedan guardadas en el notebook).
- **Usuario dedicado de mínimo privilegio** (`spark_etl`), creado en `script_mysql_bigdata.sql`, con permisos solo sobre la base `bigdata_pipeline`.
- **TLS**: `mysql-connector-python` negocia SSL automáticamente si el servidor lo soporta (`ssl_disabled=False`).
- **`autocommit=False`**: la carga es transaccional (commit/rollback explícito).

>  Antes de ejecutar esta sección, correr el **bloque 1** de `script_mysql_bigdata.sql` como `root` (crea base de datos y usuario).

In [ ]:
DB_NAME = os.getenv("MYSQL_DB", "bigdata_pipeline")

DB_CONFIG = {
    "host":               os.getenv("MYSQL_HOST", "localhost"),
    "port":               int(os.getenv("MYSQL_PORT", "3306")),
    "user":               os.getenv("MYSQL_USER", "spark_etl"),
    "password":           os.getenv("MYSQL_PASSWORD") or getpass("Contraseña MySQL: "),
    "charset":            "utf8mb4",
    "collation":          "utf8mb4_unicode_ci",
    "autocommit":         False,
    "connection_timeout": 10,
    "ssl_disabled":       False,
}

try:
    conn = mysql.connector.connect(**DB_CONFIG)
    cursor = conn.cursor()
    cursor.execute(f"CREATE DATABASE IF NOT EXISTS `{DB_NAME}` "
                   "CHARACTER SET utf8mb4 COLLATE utf8mb4_unicode_ci")
    conn.database = DB_NAME

    cursor.execute("SELECT VERSION(), CURRENT_USER(), DATABASE()")
    version, usuario, base = cursor.fetchone()
    cursor.execute("SHOW SESSION STATUS LIKE 'Ssl_cipher'")
    ssl_cipher = cursor.fetchone()[1] or "No (servidor sin TLS configurado)"

    print(f"Conexión exitosa a MySQL {version}")
    print(f"   Usuario : {usuario}")
    print(f"   Base    : {base}")
    print(f"   Cifrado : {ssl_cipher}")
except MySQLError as e:
    raise SystemExit(f"Error de conexión a MySQL: {e}")

### Paso 3.2: Modelado de las tablas de resultados
| Tabla | Clave primaria | Contenido |
|---|---|---|
| `metricas_ventas_hora` | `hora` (0–23) | Métricas agregadas por hora del día |
| `metricas_ventas_categoria` | `categoria` | Métricas agregadas por categoría de producto |

Tipos `DECIMAL` para montos (precisión exacta, sin errores de punto flotante), restricciones `CHECK` para integridad y `fecha_carga` para auditoría del pipeline.

In [ ]:
DDL_TABLAS = {
    "metricas_ventas_hora": '''
        CREATE TABLE IF NOT EXISTS metricas_ventas_hora (
            hora                  TINYINT UNSIGNED NOT NULL,
            total_transacciones   INT UNSIGNED     NOT NULL,
            clientes_unicos       INT UNSIGNED     NOT NULL,
            unidades_vendidas     INT UNSIGNED     NOT NULL,
            ticket_promedio       DECIMAL(12,2)    NOT NULL,
            ingresos_totales      DECIMAL(14,2)    NOT NULL,
            impuestos_recaudados  DECIMAL(14,2)    NOT NULL,
            fecha_carga           TIMESTAMP        NOT NULL DEFAULT CURRENT_TIMESTAMP
                                                   ON UPDATE CURRENT_TIMESTAMP,
            PRIMARY KEY (hora),
            CONSTRAINT chk_hora_rango CHECK (hora BETWEEN 0 AND 23)
        ) ENGINE=InnoDB DEFAULT CHARSET=utf8mb4 COLLATE=utf8mb4_unicode_ci
          COMMENT='Métricas de ventas completadas agregadas por hora (PySpark)'
    ''',
    "metricas_ventas_categoria": '''
        CREATE TABLE IF NOT EXISTS metricas_ventas_categoria (
            categoria             VARCHAR(50)      NOT NULL,
            total_transacciones   INT UNSIGNED     NOT NULL,
            unidades_vendidas     INT UNSIGNED     NOT NULL,
            ticket_promedio       DECIMAL(12,2)    NOT NULL,
            ticket_maximo         DECIMAL(12,2)    NOT NULL,
            ingresos_totales      DECIMAL(14,2)    NOT NULL,
            participacion_pct     DECIMAL(5,2)     NOT NULL,
            fecha_carga           TIMESTAMP        NOT NULL DEFAULT CURRENT_TIMESTAMP
                                                   ON UPDATE CURRENT_TIMESTAMP,
            PRIMARY KEY (categoria),
            CONSTRAINT chk_participacion CHECK (participacion_pct BETWEEN 0 AND 100)
        ) ENGINE=InnoDB DEFAULT CHARSET=utf8mb4 COLLATE=utf8mb4_unicode_ci
          COMMENT='Métricas de ventas completadas agregadas por categoría (PySpark)'
    ''',
    "transacciones_raw": '''
        CREATE TABLE IF NOT EXISTS transacciones_raw (
            id                INT AUTO_INCREMENT PRIMARY KEY,
            transaccion_id    VARCHAR(50),
            fecha_hora        VARCHAR(50),
            cliente_id        VARCHAR(50),
            producto          VARCHAR(100),
            categoria         VARCHAR(50),
            cantidad          INT NULL,
            precio_unitario   DECIMAL(10, 2) NULL,
            metodo_pago       VARCHAR(50),
            ciudad            VARCHAR(50),
            estado            VARCHAR(50),
            INDEX idx_tx (transaccion_id),
            INDEX idx_cliente (cliente_id),
            INDEX idx_categoria (categoria)
        ) ENGINE=InnoDB DEFAULT CHARSET=utf8mb4 COLLATE=utf8mb4_unicode_ci
          COMMENT='Dataset masivo original importado desde CSV'
    ''',
}

for nombre, ddl in DDL_TABLAS.items():
    cursor.execute(ddl)
    print(f"Tabla lista: {nombre}")
conn.commit()

cursor.execute("SHOW TABLES")
print("Tablas en la base:", [t[0] for t in cursor.fetchall()])

### Paso 3.3: Carga con sentencias preparadas (prepared statements)
- `conn.cursor(prepared=True)` → MySQL compila la sentencia **una sola vez** en el servidor y luego solo recibe los parámetros (`%s`).
- Protege contra **inyección SQL** (los valores nunca se concatenan al texto SQL).
- `ON DUPLICATE KEY UPDATE` → la carga es **idempotente**: re-ejecutar el pipeline actualiza en lugar de duplicar.
- Todo dentro de una **transacción**: si falla una fila, se hace `rollback` completo.
- Los montos se convierten a `Decimal` antes de enlazarlos, para conservar la precisión exacta de las columnas `DECIMAL`.

In [ ]:
SQL_INSERT_HORA = '''
    INSERT INTO metricas_ventas_hora
        (hora, total_transacciones, clientes_unicos, unidades_vendidas,
         ticket_promedio, ingresos_totales, impuestos_recaudados)
    VALUES (%s, %s, %s, %s, %s, %s, %s) AS nueva
    ON DUPLICATE KEY UPDATE
        total_transacciones  = nueva.total_transacciones,
        clientes_unicos      = nueva.clientes_unicos,
        unidades_vendidas    = nueva.unidades_vendidas,
        ticket_promedio      = nueva.ticket_promedio,
        ingresos_totales     = nueva.ingresos_totales,
        impuestos_recaudados = nueva.impuestos_recaudados
'''

SQL_INSERT_CATEGORIA = '''
    INSERT INTO metricas_ventas_categoria
        (categoria, total_transacciones, unidades_vendidas, ticket_promedio,
         ticket_maximo, ingresos_totales, participacion_pct)
    VALUES (%s, %s, %s, %s, %s, %s, %s) AS nueva
    ON DUPLICATE KEY UPDATE
        total_transacciones = nueva.total_transacciones,
        unidades_vendidas   = nueva.unidades_vendidas,
        ticket_promedio     = nueva.ticket_promedio,
        ticket_maximo       = nueva.ticket_maximo,
        ingresos_totales    = nueva.ingresos_totales,
        participacion_pct   = nueva.participacion_pct
'''

# Spark -> lista de tuplas Python (resultados agregados: pocas filas, collect() seguro)
cols_hora = ["hora", "total_transacciones", "clientes_unicos", "unidades_vendidas",
             "ticket_promedio", "ingresos_totales", "impuestos_recaudados"]
cols_cat = ["categoria", "total_transacciones", "unidades_vendidas", "ticket_promedio",
            "ticket_maximo", "ingresos_totales", "participacion_pct"]

def a_decimal(fila):
    return tuple(Decimal(str(v)) if isinstance(v, float) else v for v in fila)

datos_hora = [a_decimal(r) for r in metricas_hora.select(cols_hora).collect()]
datos_cat = [a_decimal(r) for r in metricas_categoria.select(cols_cat).collect()]

cursor_prep = conn.cursor()
try:
    cursor_prep.executemany(SQL_INSERT_HORA, datos_hora)
    cursor_prep.executemany(SQL_INSERT_CATEGORIA, datos_cat)
    conn.commit()
    print(f"Insertadas/actualizadas {len(datos_hora)} filas en metricas_ventas_hora")
    print(f"Insertadas/actualizadas {len(datos_cat)} filas en metricas_ventas_categoria")
except MySQLError as e:
    conn.rollback()
    print(f"Error en la carga, se aplicó ROLLBACK: {e}")
    raise
finally:
    cursor_prep.close()

### Paso 3.4: Persistencia del Dataset Crudo Masivo (`transacciones_raw`)
Para permitir consultas SQL directas y pruebas de validación sobre los 60,600 registros originales, se realiza la carga masiva de `transacciones_ecommerce.csv` en la tabla `transacciones_raw` de MySQL mediante inserción en lotes (batch processing).

In [ ]:
SQL_INSERT_RAW = '''
    INSERT INTO transacciones_raw 
    (transaccion_id, fecha_hora, cliente_id, producto, categoria, cantidad, precio_unitario, metodo_pago, ciudad, estado)
    VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s)
'''

cursor.execute("DELETE FROM transacciones_raw")
conn.commit()

batch = []
BATCH_SIZE = 5000
total_insertados = 0

with open(RUTA_CSV, mode="r", encoding="utf-8") as f:
    reader = csv.DictReader(f)
    for row in reader:
        cant_str = row["cantidad"].strip()
        cant = int(cant_str) if cant_str and cant_str.lstrip('-').isdigit() else None
        
        precio_str = row["precio_unitario"].strip()
        precio = float(precio_str) if precio_str else None

        batch.append((
            row["transaccion_id"],
            row["fecha_hora"],
            row["cliente_id"],
            row["producto"],
            row["categoria"],
            cant,
            precio,
            row["metodo_pago"],
            row["ciudad"],
            row["estado"]
        ))
        
        if len(batch) >= BATCH_SIZE:
            cursor.executemany(SQL_INSERT_RAW, batch)
            conn.commit()
            total_insertados += len(batch)
            batch = []

if batch:
    cursor.executemany(SQL_INSERT_RAW, batch)
    conn.commit()
    total_insertados += len(batch)

print(f"Total registros del CSV ingresados en 'transacciones_raw': {total_insertados:,}")

---

## Fase 4: Validación de la Persistencia

### Paso 4.1: `SELECT` final sobre MySQL
Se consulta directamente la base de datos para confirmar que los datos fueron persistidos.

In [ ]:
def consultar(sql, params=None):
    '''Ejecuta un SELECT y devuelve un DataFrame de pandas.'''
    cur = conn.cursor()
    cur.execute(sql, params or ())
    columnas = [d[0] for d in cur.description]
    filas = cur.fetchall()
    cur.close()
    return pd.DataFrame(filas, columns=columnas)

print("=== SELECT * FROM metricas_ventas_hora ===")
display(consultar("SELECT * FROM metricas_ventas_hora ORDER BY hora"))

print("=== SELECT * FROM metricas_ventas_categoria ===")
display(consultar("SELECT * FROM metricas_ventas_categoria ORDER BY ingresos_totales DESC"))

print("=== SELECT * FROM transacciones_raw (Primeros 5 registros) ===")
display(consultar("SELECT * FROM transacciones_raw LIMIT 5"))

### Paso 4.2: Conciliación Spark ↔ MySQL
Se comparan conteos y totales calculados en Spark contra los almacenados en MySQL para garantizar que no hubo pérdida ni alteración de datos.

In [ ]:
spark_tot = df_ventas.agg(
    F.count("*").alias("tx"),
    F.round(F.sum("total_con_impuestos"), 2).alias("ingresos"),
).first()

mysql_tot = consultar('''
    SELECT COUNT(*)                 AS filas,
           SUM(total_transacciones) AS tx,
           SUM(ingresos_totales)    AS ingresos
    FROM metricas_ventas_hora
''').iloc[0]

conciliacion = pd.DataFrame({
    "Métrica": ["Filas en tabla (horas)", "Transacciones", "Ingresos totales ($)"],
    "Spark":   [metricas_hora.count(), spark_tot["tx"], spark_tot["ingresos"]],
    "MySQL":   [int(mysql_tot["filas"]), int(mysql_tot["tx"]), float(mysql_tot["ingresos"])],
})
conciliacion["Coincide"] = (conciliacion["Spark"].astype(float) - conciliacion["MySQL"].astype(float)).abs() < 0.05
display(conciliacion)

print("\n=== Top 3 horas pico (consulta analítica en MySQL) ===")
display(consultar('''
    SELECT hora, total_transacciones, ingresos_totales, ticket_promedio
    FROM metricas_ventas_hora
    ORDER BY total_transacciones DESC
    LIMIT 3
'''))

### Paso 4.3: Cierre de recursos
Se liberan la conexión a MySQL y los recursos del clúster Spark.

In [ ]:
cursor.close()
conn.close()
df_enriched.unpersist()
spark.stop()
print("Conexión MySQL cerrada y SparkSession detenida. Pipeline finalizado.")

---

## Evidencia en MySQL
Capturas tomadas desde **MySQL Workbench / terminal** tras ejecutar el pipeline:

```sql
USE bigdata_pipeline;
SELECT * FROM metricas_ventas_hora ORDER BY hora;
SELECT * FROM metricas_ventas_categoria ORDER BY ingresos_totales DESC;
```

<!-- Reemplazar con las capturas reales -->
![Evidencia tabla metricas_ventas_hora](capturas/mysql_metricas_hora.png)
![Evidencia tabla metricas_ventas_categoria](capturas/mysql_metricas_categoria.png)

---

## Conclusiones
1. **Escalabilidad:** el mismo código PySpark procesa 60K o 60M de filas; solo cambia el `master` (local → clúster YARN/K8s). El esquema explícito, AQE y la caché reducen el costo de cómputo.
2. **Calidad de datos:** de 60,600 filas se descartaron 3,342 (**~5.5 %**: nulos críticos, duplicados, cantidades/precios inconsistentes), quedando 57,258 registros limpios y 45,816 ventas completadas. Sin esta limpieza, las métricas de ingresos estarían sesgadas.
3. **Patrón de negocio:** las ventas se concentran en el **mediodía (12 h: 3,533 tx)** y en el **pico nocturno (20 h: 3,793 tx, la hora más rentable)**; *Electrónica* aporta el **53.9 %** de los ingresos con apenas el 20 % de las transacciones, lo que orienta campañas y capacidad de infraestructura en esas franjas.
4. **Persistencia confiable:** las sentencias preparadas + transacción + `ON DUPLICATE KEY UPDATE` hacen la carga **segura, atómica e idempotente**, y la conciliación Spark ↔ MySQL confirma la integridad de los datos.